# 07_05 English and Greek: the same steps in Butler's English

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → [numbers] → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **numbers**
>
> Project files this connects to:
>
> - `data/homer/odyssey/english/`
> - `src/homer_ai/tokenizer.py`
>
> **This notebook:** The same code can learn Butler's English too. How do its characters, line lengths and token counts differ from Greek's?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How the same code handles Butler's English, and how its vocabulary compares with Greek's
- Why prose (long paragraph lines) differs from verse
- How many tokens characters and GPT-2's tokenizer use for English and for Greek
- How a tiny English model trains and writes

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **verse** | writing in short lines with a rhythm: the Greek poems have one verse per line | `μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος` |
| **prose** | ordinary writing in sentences and paragraphs: Butler's translation | one paragraph per line in `data/homer/*/english/` |

## 3. The idea

### ❓ Can the same code learn English?

Yes. The code never knew the text was Greek. `CharTokenizer` takes whatever characters it finds. Here's Butler's Book 1 of the Odyssey, next to the Greek Book 1, with the vocabulary of each (the number of different characters, plus our 2 special tokens).

👉 **Predict first:** English has far fewer different characters than Greek (about 67 against 167, see [07_01](07_01_the_corpus.ipynb)). Will Book 1 alone follow the same pattern?

In [ ]:
# data/homer: load_book_files reads all the books of a folder
from homer_ai.lessons import HOMER, load_book_files

In [ ]:
import math

import torch
import torch.nn.functional as F

from homer_ai.model import HomerGreekLLM
from homer_ai.tokenizer import CharTokenizer

greek = load_book_files(HOMER / "odyssey" / "greek")
english = load_book_files(HOMER / "odyssey" / "english")

for name, books in (("Greek", greek), ("English", english)):
    book_1 = books[0]
    whole = CharTokenizer("\n".join(books))
    in_poem = len(set("".join(books)))
    print(
        f"{name:8} book 1: {len(book_1):,} characters, {len(set(book_1))} different; "
        f"whole poem: {in_poem} different, vocabulary {whole.vocab_size}"
    )

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Book 1 uses 135 different characters in Greek and 64 in English. The whole poem uses 167 and 67. Few new letters appear after Book 1, because the whole alphabet and punctuation show up early. The English vocabulary (69 with the special tokens) is less than half the Greek (169).

</details>

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: what's different about prose? (lines are paragraphs)

Greek verse has one verse per line, and English prose has one **paragraph** per line. So in English, line breaks are much rarer, and a "line" is much longer. What does that mean for a model that learns which letter comes next?

👉 **Predict first:** the model reads 24 characters at a time. In Greek a verse is about 44 characters. Roughly how often does a line break (`⏎`) come up in each text?

In [ ]:
for name, books in (("Greek", greek), ("English", english)):
    book_1 = books[0]
    lines = book_1.splitlines()
    breaks = book_1.count("\n")
    every = len(book_1) / breaks
    print(f"{name:8} {len(lines):>3} lines; a break every {every:.0f} characters")

print()
print("Greek line 1:  ", greek[0].splitlines()[0])
paragraph = english[0].splitlines()[0]
print("English line 1:", paragraph[:60], "…", f"({len(paragraph)} characters)")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

One break every 44 characters in Greek, and one every 665 in English. So a model with a 24-character window almost never sees a break in English, while the Greek model sees one about every other window. The first English line is a whole paragraph of 605 characters, not a verse. Butler's paragraphs don't match Greek verses one to one, which is why we can't pair them line by line.

</details>

### ❓ Step 2: how many tokens do characters and GPT-2's tokenizer use for each language?

Our tokenizer gives one token per character, always. GPT-2's tokenizer (via `tiktoken`, [A_05](../A_toolbox/A_05_tiktoken_bpe.ipynb)) is a BPE ([02_03](../02_text_to_numbers/02_03_bpe.ipynb)) that learned its merges from mostly English text. We count tokens for the first 1,000 characters of Book 1 in each language, and for the Greek verse and the English paragraph. (The first time, `tiktoken` downloads GPT-2's vocabulary file, so this needs an internet connection once.)

👉 **Predict first:** for English, will GPT-2 use more or fewer tokens than characters? And for Greek? ([A_05](../A_toolbox/A_05_tiktoken_bpe.ipynb) found Greek needing more tokens than letters.)

In [ ]:
import tiktoken

# the first time, this downloads GPT-2's vocabulary file (needs internet once)
gpt2 = tiktoken.get_encoding("gpt2")

samples = {
    "Greek, first 1,000 characters": greek[0][:1000],
    "English, first 1,000 characters": english[0][:1000],
    "Greek, verse 1": greek[0].splitlines()[0],
    "English, paragraph 1": english[0].splitlines()[0],
}
print(f"{'text':33} {'characters':>10} {'GPT-2 tokens':>13} {'chars per token':>16}")
counts = {}
for name, text in samples.items():
    tokens = len(gpt2.encode(text))
    counts[name] = tokens
    print(f"{name:33} {len(text):>10,} {tokens:>13,} {len(text) / tokens:>16.2f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

1,000 English characters become 227 GPT-2 tokens (4.41 characters per token), but 1,000 Greek characters become **1,284** tokens (0.78 characters per token). That's more tokens than letters. The single Greek verse is 50 characters and 57 tokens. GPT-2's merges were learned from mostly English text, so for Greek it often falls back to pieces of single letters ([A_05](../A_toolbox/A_05_tiktoken_bpe.ipynb)).

</details>

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import BAD, GOOD, INK, MUTED, studio_axes, studio_figure

names = ["our tokenizer\n(any text)", "GPT-2 on\nEnglish", "GPT-2 on\nGreek"]
values = [
    1000,
    counts["English, first 1,000 characters"],
    counts["Greek, first 1,000 characters"],
]
fig = studio_figure("tokens", "Tokens needed for 1,000 characters", (9, 5.5))
ax = studio_axes(fig, (0.1, 0.12, 0.8, 0.64))
ax.bar(names, values, color=[MUTED, GOOD, BAD], width=0.55)
for x, value in enumerate(values):
    ax.text(x, value + 25, f"{value:,}", ha="center", color=INK,
            family="monospace", fontsize=14, fontweight="bold")
ax.tick_params(axis="x", labelsize=12, labelcolor=INK)
ax.set_ylabel("tokens")
ax.set_ylim(0, 1500)
ax.grid(axis="x", visible=False)
ax.set_axisbelow(True)
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Our tokenizer is the grey bar, always 1,000 for 1,000 characters. GPT-2 is much shorter on English (green) and much longer on Greek (pink-red). So which tokenizer is better depends on the language. A BPE saves a lot on English, but GPT-2's BPE is worse than plain characters for Greek. A BPE trained on Greek is one of the ideas at the end of this notebook. (Mind that this counts tokens only. It doesn't measure quality.)

</details>

### ❓ Step 3: can a tiny model learn English?

We follow the same steps as for the Iliad in [07_02](07_02_an_iliad_model.ipynb): a tokenizer from Butler's Book 1, the first 10,000 characters, a 90/10 split ([05_05](../05_training_on_the_odyssey/05_05_validation.ipynb)), the model, 6 epochs, a fixed seed. Nothing in `HomerGreekLLM` changes (the name says Greek, but nothing inside it is Greek-specific).

👉 **Predict first:** the vocabulary is 66, so a random guess costs `ln(66)`. Will the trained loss be higher or lower than Greek's loss near 3 in [07_02](07_02_an_iliad_model.ipynb)?

In [ ]:
from homer_ai.lessons import CHUNK, evaluate  # 05_05

tokenizer = CharTokenizer(english[0])  # only Book 1: a small, quick vocabulary
data = torch.tensor(tokenizer.encode(english[0][:10000]))
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]


torch.manual_seed(42)
model = HomerGreekLLM(vocab_size=tokenizer.vocab_size, embed_dim=32)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
guess = math.log(tokenizer.vocab_size)
print(f"vocabulary {tokenizer.vocab_size}; random guessing = ln(66) = {guess:.3f}")
print(f"untrained validation loss: {evaluate(model, val_data):.3f}")

for epoch in range(6):
    for i in range(0, len(train_data) - CHUNK, CHUNK):
        inputs = train_data[i : i + CHUNK]
        targets = train_data[i + 1 : i + CHUNK + 1]
        loss = F.cross_entropy(model(inputs), targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss = evaluate(model, train_data)
    val_loss = evaluate(model, val_data)
    print(f"epoch {epoch + 1} | train {train_loss:.3f} | validation {val_loss:.3f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The untrained loss is 4.208, near `ln(66) = 4.190` (pure guessing). After six epochs the training loss is 2.411 and the validation loss 2.404. They're close together and still falling, so it learned real letter patterns and didn't memorise. It ends lower than Greek's (about 2.8 in 07_02), but see the next question before you conclude anything from that.

</details>

### ❓ What does the English model write?

We use the same top-k writing as [06_02](../06_making_it_talk/06_02_less_repetition.ipynb), from `Tell me, O Muse` (Butler's first words): 120 new characters, top-k 5 and temperature 0.8, with greedy for comparison.

👉 **Predict first:** will it write English words, or word-shaped pieces?

In [ ]:
from homer_ai.lessons import write  # the function of 06_02; top_k=1 is greedy

print("greedy:", write(model, tokenizer, "Tell me, O Muse", 120, top_k=1))
print("top-5: ", write(model, tokenizer, "Tell me, O Muse", 120))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Greedy loops at once (`an an an …`, like the Greek `και και και`). Top-5 writes English-shaped pieces with a few real small words (`and`, `an`, `or`): `…Musend therisof angin ou an ther illo and,ofe athar har and tha …`. So it learned letter patterns like `th`, `an`, `er`, but no sentences. Same as for Greek.

</details>

### ❓ Can I compare this English loss with the Greek loss?

No. The vocabularies differ (66 against 175 or 171), the texts differ, and the held-out parts are different texts. That's exactly the unfairness of [07_03](07_03_comparing_fairly.ipynb). A lower English loss only says that guessing the next letter among fewer, simpler characters is easier. It doesn't say English is "better".

### ❓ Why does Butler write `Ulysses` and `Minerva`?

Butler used the Roman (Latin) names: `Ulysses` is the Roman name for Odysseus, and `Minerva` the Roman name for Athena. A character-level model trained on his English never sees the Greek names, and the Greek model never sees the Roman ones.

## 5. Recap

- The same code reads English. The vocabulary is much smaller (67 characters against 167), and prose has long lines, so line breaks are rare.
- GPT-2's tokenizer needs far fewer tokens for English (about 4.4 characters per token), but more tokens than letters for Greek.
- A tiny model learns English letter patterns, not sentences. Its loss can't be compared with Greek's (different vocabulary and text).

## You finished the course

### ❓ What did I build?

Starting from nothing but Homer's text, you built every part of a small language model yourself:

- A tokenizer that turns Greek into numbers and back (chapter 2).
- Attention and the parts of a GPT-style model (chapters 3 and 4).
- A training loop, with loss, learning rate and a held-out check (chapter 5).
- A model that writes Greek, and an MCP tool that lets Claude call it (chapter 6).
- An Iliad model, a fair comparison of two models, and one model for both poems (this chapter).

### ❓ What can I do now?

- Explain, step by step, how an LLM turns text into a next-token guess, and why it needs training.
- Read a training run: a loss that falls, one that jumps, and one that only memorises.
- Run a fair experiment: change one thing, fix the seed, test on text the model hasn't seen.
- Offer your own model to an AI app as a tool.

Yours is a toy (835,072 weights in the shipped GPT, 11,840 in the small model, against GPT-2 small's 124 million). But it's built the same way, and you know where every part sits.

*When I build chatbots for regulated teams, the first question is always: where did that answer come from? Keep asking it of everything you build.*

### ❓ What could I try myself?

Here are three things you can do with the code you have:

1. **A BPE tokenizer trained on Greek.** One character per token makes lines long and tokens weak. You built a small BPE in [02_03](../02_text_to_numbers/02_03_bpe.ipynb); training one on `data/homer/` (instead of using GPT-2's) would give Greek-sized tokens.
2. **Make the GPT bigger.** The shipped GPT ([04_05](../04_building_the_model/04_05_the_full_model.ipynb), `src/homer_ai/gpt.py`) is small on purpose, to train in minutes. Change `GPT_SETTINGS` in `train_homer.py` (wider, more blocks) and compare held-out loss fairly, as in [05_04](../05_training_on_the_odyssey/05_04_running_fair_experiments.ipynb). With more text, a bigger model can pay off.
3. **Train on the whole Odyssey (and the Iliad).** We used one book at a time. All 24 books give about 540,000 characters, which needs longer runs (see [07_02](07_02_an_iliad_model.ipynb)) and, probably, the bigger GPT of idea 2.

And there are three more advanced chapters if you want them. [Chapter 8](../08_advanced_fine_tuning/08_00_chapter_overview.ipynb) loads GPT-2's 124 million weights into the `GPTModel` you built, and fine-tunes it for a new job. [Chapter 9](../09_advanced_grounded_answers/09_00_chapter_overview.ipynb) answers questions about Homer from the real text, with the book and line beside each answer. [Chapter 10](../10_advanced_your_domain/10_00_chapter_overview.ipynb) takes the same pattern to a company's own documents, emails and tickets, on invented data.

Stuck on a tool? The [toolbox](../A_toolbox/A_00_overview.ipynb) has it. Want the whole map again? [The course map](../00_start_here/00_00_course_map.ipynb). And if you haven't read it: Sebastian Raschka's book (see the [README](../../README.md)) goes much further than we did.